# PXP ScarFinder on a 32-site chain

This notebook shows a minimal ScarFinder setup for the open-boundary PXP model at `L = 32`. The projected evolution uses `Delta t = 0.1`, while a separate fine-grained TEBD evolution with `dt = 0.01` is used for a short diagnostic trace after the ScarFinder loop.

## Imports

We keep the example close to the public package surface: build the local 3-site PXP term, assemble a three-layer brick schedule, run the projected loop, and then probe the final state with a short unprojected TEBD diagnostic.

In [ ]:
using MPSToolkit
using ITensors
using ITensorMPS
using LinearAlgebra
using Statistics

## Helper functions

The PXP Hamiltonian is a sum of 3-site terms `P_{j-1} X_j P_{j+1}`. In this notebook the neighbors are projected onto `|Dn>` and the chain is evolved with a three-color schedule over the left indices of each 3-site gate.

In [ ]:
function pxp_local_hamiltonian()
    projector_dn = ComplexF64[0 0; 0 1]
    pauli_x = ComplexF64[0 1; 1 0]
    return kron(projector_dn, kron(pauli_x, projector_dn))
end

function pxp_schedule(nsites::Int)
    starts = Int[]
    for offset in 1:3
        append!(starts, offset:3:(nsites - 2))
    end
    return starts
end

function pxp_mpo(sites)
    opsum = OpSum()
    for j in 2:(length(sites) - 1)
        opsum += 2.0, "ProjDn", j - 1, "Sx", j, "ProjDn", j + 1
    end
    return MPO(opsum, sites)
end

function z2_state(sites)
    return MPS(sites, n -> isodd(n) ? "Up" : "Dn")
end

function anti_z2_state(sites)
    return MPS(sites, n -> isodd(n) ? "Dn" : "Up")
end

function revival_history(reference_a, reference_b, psi, evolution, nsteps)
    state = deepcopy(psi)
    overlaps = Float64[]
    for _ in 1:nsteps
        evolve!(state, evolution)
        push!(overlaps, max(abs2(inner(reference_a, state)), abs2(inner(reference_b, state))))
    end
    return overlaps
end

## Set up the `L = 32` ScarFinder and TEBD evolutions

The projected loop uses `Delta t = 0.1`, `nstep = 200`, and TEBD truncation `maxdim = 64`. A separate diagnostic evolution uses `dt = 0.01` so we can probe the final state with a finer time grid.

In [ ]:
nsites = 32
sites = siteinds("S=1/2", nsites)
schedule = pxp_schedule(nsites)
local_hamiltonian = pxp_local_hamiltonian()
hamiltonian_mpo = pxp_mpo(sites)

scar_evolution = tebd_evolution_from_hamiltonians(
    fill(local_hamiltonian, length(schedule)),
    0.1;
    schedule=schedule,
    nstep=1,
    maxdim=64,
    cutoff=1e-10,
)

diagnostic_evolution = tebd_evolution_from_hamiltonians(
    fill(local_hamiltonian, length(schedule)),
    0.01;
    schedule=schedule,
    nstep=1,
    maxdim=64,
    cutoff=1e-10,
)

truncation = BondDimTruncation(1; cutoff=1e-10)
energy_target = EnergyTarget(0.0; operator=hamiltonian_mpo, tol=1e-8, alpha=0.1, maxstep=4)

z2 = z2_state(sites)
anti_z2 = anti_z2_state(sites)
psi = deepcopy(z2)

println("schedule length = ", length(schedule))
println("initial bond entropy = ", bond_entropy(psi, 16))

## Warm up the compiled kernels

Three-site gate application compiles the first time it is used. Warming up once makes the actual benchmark cells much more representative of steady-state runtime.

In [ ]:
warmup_state = deepcopy(psi)
evolve!(warmup_state, diagnostic_evolution)
scarfinder_step!(warmup_state, scar_evolution, truncation; target_energy=energy_target)
nothing

## Run the 200-step ScarFinder loop

We call `scarfinder_step!` directly so we can record simple convergence diagnostics: overlap between consecutive iterates, overlap with the `|Z2>` seed, and the targeted energy density.

In [ ]:
step_overlap = Float64[]
z2_overlap = Float64[]
energy_history = Float64[]

for _ in 1:200
    previous = deepcopy(psi)
    scarfinder_step!(psi, scar_evolution, truncation; target_energy=energy_target)
    push!(step_overlap, abs2(inner(previous, psi)))
    push!(z2_overlap, abs2(inner(z2, psi)))
    push!(energy_history, energy_density(psi, hamiltonian_mpo))
end

println("final |Z2> overlap = ", z2_overlap[end])
println("final energy density = ", energy_history[end])
println("final bond entropy = ", bond_entropy(psi, 16))
println("final maxlinkdim = ", maxlinkdim(psi))
println("mean step fidelity over last 20 steps = ", mean(step_overlap[end-19:end]))

## Short fine-TEBD diagnostic

This is not a full scar-validation study, but it is a quick way to inspect whether the final state stays near the `|Z2>` / `|Z2bar>` revival family over a short exact-TEBD window with `dt = 0.01`.

In [ ]:
revivals = revival_history(z2, anti_z2, psi, diagnostic_evolution, 100)

println("peak |Z2>/<Z2bar>| overlap over 100 fine TEBD steps = ", maximum(revivals))
println("last-step revival-family overlap = ", revivals[end])

## Notes

- This notebook is meant as a runnable PXP ScarFinder setup, not as a complete optimization study.
- The explicit `EnergyTarget` keeps the projected loop near the zero-energy scar sector of the open PXP chain.
- If you want to explore a broader variational family, the next knob to turn is the projection rank in `BondDimTruncation`.